# Your RMSE was a lie

**Lecture 2 · Fix** · Géron, Chapters 2 & 4 · *Mathematical thread: least squares and the normal equation*

Applications of Machine Learning — BSc Mathematics of Artificial Intelligence

---

**How to use this notebook.** You are not expected to type the code. You are
expected to *read* it before you run it, and to be able to say what every line
does and what would break if it changed. Cells marked **⚠ read before running**
contain a defect on purpose.

Run the cells in order. Anything that takes more than a few seconds says so.

**About the prompt boxes.** Every code cell in this notebook is preceded by a
quoted prompt, and three lines follow it: what the prompt leaves open, the
version a student typically writes instead, and how you would catch a wrong
answer. Those three lines are the part worth reading twice.

The prompts here are **specifications, not transcripts** — this is what you
would have to ask for in order to get this cell, not a recording of somebody
asking for it. If your own prompt is vaguer than the box, expect worse code than
the cell below it.

*Lecture 19 is the one exception in this course.* It was built cell by cell
against Colab's Gemini 3.1 Pro, and its prompts are verbatim. It says so itself.


## 1 · Setup and where we left off

> **Prompt · setup**
>
> **input** · nothing
>
> **output** · the version of every library this notebook depends on, and one seed
>
> **constraint** · ASSERT the scikit-learn version rather than printing it — `root_mean_squared_error` arrived in 1.4, and on an older Colab image the failure is an ImportError twenty cells from here

**Watch this prompt.**

* **Left open:** that RANDOM_STATE is defined once and used for every split, every model and every shuffle. A notebook with three different seeds in it cannot be reproduced by reading it.
* **The usual student version:** printing the versions and not checking them, so the notebook reports its own incompatibility as information rather than as an error.
* **How you would catch it:** not examinable, and it is here because a version mismatch produces a confusing error in a cell that has nothing to do with versions.

In [ ]:
# --- setup -------------------------------------------------------------------
# Not examinable: this is engineering hygiene, not machine learning. It is here
# because a version mismatch produces a confusing error twenty cells later.
import sys, sklearn, numpy as np, pandas as pd, matplotlib

print(f"python       {sys.version.split()[0]}")
print(f"scikit-learn {sklearn.__version__}")
print(f"numpy        {np.__version__}")
print(f"pandas       {pd.__version__}")

# root_mean_squared_error arrived in scikit-learn 1.4
assert tuple(int(p) for p in sklearn.__version__.split(".")[:2]) >= (1, 4), \
    "This notebook needs scikit-learn >= 1.4.  In Colab: %pip install -U scikit-learn"

RANDOM_STATE = 42          # every split, every model, every shuffle
pd.set_option("display.width", 100)

> **Prompt · the data**
>
> **input** · the California housing tarball
>
> **output** · 20,640 districts and 10 columns
>
> **constraint** · a FUNCTION that downloads if absent and reads if present — the data will change, and you will need this on another machine
>
> **check** · assert the shape, rather than trusting the download

**Watch this prompt.**

* **Left open:** what to do if the download is truncated. A short read gives a smaller frame and the assert catches it; anything subtler it will not.
* **The usual student version:** downloading by hand and reading a path under ~/Downloads. It works on your machine and nowhere else, which you discover at the demo.
* **How you would catch it:** delete `datasets/` and re-run. If the cell cannot rebuild its own input from nothing, it is not reproducible, it is cached.

In [ ]:
# --- the data ----------------------------------------------------------------
# A function, not a manual download: the data will change, and you will need
# this on another machine.  ~5 s the first time, instant afterwards.
from pathlib import Path
import tarfile, urllib.request

def load_housing():
    tarball = Path("datasets/housing.tgz")
    if not tarball.is_file():
        Path("datasets").mkdir(parents=True, exist_ok=True)
        url = "https://github.com/ageron/data/raw/main/housing.tgz"
        urllib.request.urlretrieve(url, tarball)
        with tarfile.open(tarball) as t:
            t.extractall(path="datasets", filter="data")
    return pd.read_csv("datasets/housing/housing.csv")

housing_full = load_housing()

assert housing_full.shape == (20640, 10), f"unexpected shape {housing_full.shape}"
print(f"{len(housing_full):,} districts, {housing_full.shape[1]} columns")
housing_full.head()

> **Prompt · every import, and the same split**
>
> **input** · the same data and the same seed
>
> **output** · the identical 16,512 / 4,128 split as the previous lecture
>
> **constraint** · every import this notebook needs in ONE place, and the split rebuilt from the seed rather than inherited
>
> **check** · assert the two sizes exactly — if they differ, every comparison against the previous lecture is void

**Watch this prompt.**

* **Left open:** that the stratification bins are repeated here verbatim. Change them and you get a different split from the same seed.
* **The usual student version:** continuing in the previous notebook's kernel, where all of this already exists. It works in the room and nowhere else.
* **How you would catch it:** a notebook that only runs because a previous one is still in memory is not reproducible. Restart-and-run-all is the only test of that.

In [ ]:
# Every import this notebook needs, in one place — a notebook that only runs
# because a previous one is still in memory is not reproducible.
from sklearn.compose import ColumnTransformer
from sklearn.ensemble import RandomForestRegressor
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LinearRegression
from sklearn.metrics import root_mean_squared_error
from sklearn.model_selection import (GridSearchCV, KFold, cross_val_score,
                                     train_test_split)
from sklearn.pipeline import Pipeline, make_pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler, add_dummy_feature
from sklearn.tree import DecisionTreeRegressor
import matplotlib.pyplot as plt

income_cat = pd.cut(housing_full["median_income"],
                    bins=[0., 1.5, 3.0, 4.5, 6., np.inf], labels=[1, 2, 3, 4, 5])
train_set, test_set = train_test_split(
    housing_full, test_size=0.2, random_state=RANDOM_STATE, stratify=income_cat)

housing  = train_set.copy()
X_train  = housing.drop(columns=["median_house_value"])
y_train  = housing["median_house_value"]
X_test   = test_set.drop(columns=["median_house_value"])
y_test   = test_set["median_house_value"]

assert len(X_train) == 16512 and len(X_test) == 4128
print("same split as the previous lecture — the seed guarantees it")

## 2 · Thread 1 — what `LinearRegression().fit()` actually computed

Minimising $\lVert X\theta - y\rVert^2$ gives the normal equation

$$X^{\mathsf T}(X\hat\theta - y) = 0$$

Read row by row, that says the residual is orthogonal to **every column of X**.
Least squares is not an algebraic trick — it is a projection onto the column
space of $X$. Verify it rather than believing it:

> **Prompt · what LinearRegression().fit() actually computed**
>
> **input** · the numeric features with an intercept column added
>
> **output** · the normal-equation solution, and the residual's inner product with every column of X
>
> **constraint** · add the intercept column with `add_dummy_feature` BEFORE solving — without it the residual is not orthogonal to the constant and the assert fails for the wrong reason
>
> **check** · assert the largest |Xᵀ(Xθ̂ − y)| is negligible RELATIVE to the scale of y — an absolute tolerance on dollars is meaningless

**Watch this prompt.**

* **Left open:** what the orthogonality means. Read row by row, Xᵀ(Xθ̂ − y) = 0 says the residual is orthogonal to EVERY COLUMN of X: least squares is not an algebraic trick, it is a projection onto the column space.
* **The usual student version:** taking the normal equation on faith. It is four lines to verify and the verification is the thread.
* **How you would catch it:** `np.linalg.inv` is not what scikit-learn uses. It computes the pseudoinverse via SVD, which still returns an answer when XᵀX is singular — more features than instances, or two collinear columns. That is the whole failure condition.

In [ ]:
num = X_train.select_dtypes(include=[np.number])
X = make_pipeline(SimpleImputer(strategy="median"), StandardScaler()).fit_transform(num)
X_b = add_dummy_feature(X)                     # the x0 = 1 column, for the intercept
y = y_train.values

theta = np.linalg.inv(X_b.T @ X_b) @ X_b.T @ y
residual = X_b @ theta - y

# every column of X is orthogonal to the residual, to numerical precision
orth = X_b.T @ residual
print(f"largest |Xᵀ(Xθ̂ − y)| = {np.abs(orth).max():.3e}")
print(f"relative to the scale of y ({np.abs(y).mean():,.0f}): "
      f"{np.abs(orth).max() / np.abs(y).mean():.2e}")
assert np.abs(orth).max() / np.abs(y).mean() < 1e-6, "not orthogonal — check X"

`np.linalg.inv` is not what scikit-learn uses. It computes the pseudoinverse via
SVD, which still returns an answer when $X^{\mathsf T}X$ is singular — when you
have more features than instances, or when two columns are collinear. That is
the whole failure condition, and it is why you were warned against engineering a
feature as a weighted sum of existing ones.

## 3 · Why the tree scored zero

Not because it is perfect. Because we asked it to grade its own homework.

> **Prompt · why the tree scored zero**
>
> **input** · the same pipeline and the same training rows
>
> **output** · its RMSE on the data it was fitted to
>
> **constraint** · score it on the training rows again, and say in the output what that means

**Watch this prompt.**

* **Left open:** that it is not zero because the tree is perfect. It is zero because we asked it to grade its own homework, and an unconstrained tree can put every training row in its own leaf.
* **The usual student version:** concluding the tree is the best model, or concluding it is broken. Neither: it is a correct answer to a question nobody should have asked.
* **How you would catch it:** any model flexible enough to memorise will score perfectly on its own training rows. A zero training error is a statement about capacity, not about accuracy.

In [ ]:
num_cols = X_train.select_dtypes(include=[np.number]).columns.tolist()
preprocessing = ColumnTransformer([
    ("num", make_pipeline(SimpleImputer(strategy="median"), StandardScaler()), num_cols),
    ("cat", OneHotEncoder(handle_unknown="ignore"), ["ocean_proximity"]),
])

tree = Pipeline([("prep", preprocessing),
                 ("model", DecisionTreeRegressor(random_state=RANDOM_STATE))])
tree.fit(X_train, y_train)
print(f"RMSE on the data it was fitted to: "
      f"${root_mean_squared_error(y_train, tree.predict(X_train)):,.0f}")
print("An unconstrained tree can put every training row in its own leaf.")

## 4 · Measure it honestly

`shuffle=True` is not decoration. The default `KFold` does **not** shuffle, so
two students whose dataframes are in different row orders get different folds
and cannot work out why their numbers disagree.

⏱ **about 90 seconds** — thirty fits in total, ten of them forests.

> **Prompt · ⏱ 90 s — measure it honestly**
>
> **input** · the three models, ten folds each
>
> **output** · mean, standard deviation and range of the fold RMSEs
>
> **constraint** · `shuffle=True` is NOT decoration — the default KFold does not shuffle, so two students whose dataframes are in different row orders get different folds and cannot work out why their numbers disagree

**Watch this prompt.**

* **Left open:** how to read the spread. The folds span several thousand dollars, so any comparison that turns on less than a couple of thousand is not a comparison.
* **The usual student version:** reporting only the mean. A mean of $50,000 built from folds spanning $8,000 supports very different claims from one built from folds spanning $500.
* **How you would catch it:** print the fold minimum and maximum beside the mean, every time. It is one f-string and it decides which differences you are allowed to talk about.

In [ ]:
cv = KFold(n_splits=10, shuffle=True, random_state=RANDOM_STATE)

models = {
    "Linear regression": LinearRegression(),
    "Decision tree":     DecisionTreeRegressor(random_state=RANDOM_STATE),
    "Random forest":     RandomForestRegressor(n_estimators=100,
                                               random_state=RANDOM_STATE, n_jobs=-1),
}

results = {}
for name, model in models.items():
    pipe = Pipeline([("prep", preprocessing), ("model", model)])
    folds = -cross_val_score(pipe, X_train, y_train, cv=cv,
                             scoring="neg_root_mean_squared_error")
    results[name] = folds
    print(f"{name:20s} ${folds.mean():>9,.0f}  ± ${folds.std():>6,.0f}"
          f"   (folds ${folds.min():,.0f} – ${folds.max():,.0f})")

Report the spread, not just the mean. The folds span several thousand dollars,
so **any comparison that turns on less than a couple of thousand is not a
comparison.**

Compare the models on the *same* folds rather than comparing two averages —
paired differences remove the fold-to-fold variation that both models share:

> **Prompt · compare on the SAME folds**
>
> **input** · the two arrays of per-fold scores
>
> **output** · the paired difference, and how many folds each model wins
>
> **constraint** · subtract PER FOLD rather than comparing two averages — paired differences remove the fold-to-fold variation that both models share

**Watch this prompt.**

* **Left open:** why the paired standard deviation is so much smaller than either model's own. The folds differ in difficulty and both models feel it identically, so the difference cancels it.
* **The usual student version:** comparing two means with their own standard deviations, concluding the intervals overlap, and declaring the comparison inconclusive. The paired version usually is not.
* **How you would catch it:** report the win count. '10 of 10 folds' is an argument that a mean difference with a large standard deviation is not.

In [ ]:
diff = results["Random forest"] - results["Linear regression"]
print(f"forest − linear, per fold:  mean ${diff.mean():,.0f}  sd ${diff.std():,.0f}")
print(f"folds where the forest wins: {(diff < 0).sum()}/10")

## 5 · Tune — on validation folds, never on the test set

⏱ **2–4 minutes.** Fifteen combinations × five folds = 75 forest fits. The
lecture's figure uses `cv=10`, which takes twice as long; five is enough here.

> **Prompt · ⏱ 2-4 min — tune on validation folds, never on the test set**
>
> **input** · fifteen combinations, five folds each
>
> **output** · the best parameters and the best cross-validated RMSE
>
> **constraint** · the grid searches the WHOLE PIPELINE, so the preprocessing is refitted inside every fold — `model__` prefixes because the parameters belong to a step
>
> **check** · detect whether the winner sits on the EDGE of the grid and say so — an optimum at the boundary means the optimum may lie outside it

**Watch this prompt.**

* **Left open:** that `cv=5` here where the deck uses 10. Five is enough to choose between these fifteen and it halves the wall clock; the choice is stated rather than silent.
* **The usual student version:** reading `best_score_` as the model's accuracy. It is the score of the winner of a fifteen-way selection, measured on the folds that selected it, and it is optimistic by construction.
* **How you would catch it:** the edge-of-grid check is four lines and it is the difference between a search and a shrug. If the largest value wins, the search was too small.

In [ ]:
grid = {"model__max_features": [4, 6, 8, 10, 12],
        "model__n_estimators": [30, 100, 200]}

search = GridSearchCV(
    Pipeline([("prep", preprocessing),
              ("model", RandomForestRegressor(random_state=RANDOM_STATE, n_jobs=-1))]),
    grid, cv=5, scoring="neg_root_mean_squared_error", n_jobs=-1)
search.fit(X_train, y_train)

print(f"best {search.best_params_}")
print(f"best cross-validated RMSE ${-search.best_score_:,.0f}")

best_n = search.best_params_["model__n_estimators"]
if best_n == max(grid["model__n_estimators"]):
    print("\n⚠ the winner sits on the EDGE of the grid — the optimum may lie "
          "outside it. Search again with larger values.")

## 6 · Look at what it gets wrong

Two things the previous lecture promised and never did. Both take one cell.

> **Prompt · look at what it gets wrong**
>
> **input** · the tuned model's training predictions
>
> **output** · RMSE broken down by ocean proximity and by income band
>
> **constraint** · break the error down by GROUP — a single RMSE is an average over districts that are not alike

**Watch this prompt.**

* **Left open:** which group is about to matter. ISLAND — five districts in the whole state — is the category the first lecture warned you about.
* **The usual student version:** reporting the headline RMSE and stopping. The stakeholder will ask 'is it worse anywhere in particular', and this is the cell that answers it.
* **How you would catch it:** `observed=True` on the groupby. Without it pandas produces a row for every unobserved combination of categories, full of NaN, and the table becomes unreadable.

In [ ]:
best = search.best_estimator_
pred_train_cv = best.predict(X_train)

err = pd.DataFrame({
    "actual": y_train,
    "predicted": pred_train_cv,
    "error": pred_train_cv - y_train,
    "ocean": housing["ocean_proximity"],
    "income_cat": pd.cut(housing["median_income"],
                         bins=[0., 1.5, 3.0, 4.5, 6., np.inf], labels=[1, 2, 3, 4, 5]),
})

print("RMSE by ocean_proximity:")
print(err.groupby("ocean", observed=True)["error"]
        .apply(lambda e: np.sqrt((e ** 2).mean())).sort_values(ascending=False)
        .apply(lambda v: f"${v:,.0f}"))

print("\nRMSE by income category:")
print(err.groupby("income_cat", observed=True)["error"]
        .apply(lambda e: np.sqrt((e ** 2).mean()))
        .apply(lambda v: f"${v:,.0f}"))

`ISLAND` — five districts in the whole state — is the category the first lecture
warned you about. With ten folds, some folds contain no ISLAND training row at
all, and `handle_unknown="ignore"` then encodes it as an all-zero column and
says nothing.

> **Prompt · what an unseen category actually encodes to**
>
> **input** · an encoder fitted without ISLAND, asked to transform ISLAND
>
> **output** · the resulting row, and its sum
>
> **constraint** · DEMONSTRATE it rather than describing it — fit without the category and transform with it

**Watch this prompt.**

* **Left open:** the consequence for cross-validation. With ten folds, some folds contain no ISLAND training row at all, and `handle_unknown='ignore'` then encodes it as an all-zero column and says nothing.
* **The usual student version:** setting `handle_unknown='ignore'` because the alternative raises, and never finding out what it does instead. It is the right choice and it is silent.
* **How you would catch it:** the sum of the encoded row is zero. Every other district's row sums to one, so the model sees ISLAND as 'none of the above' — which is a prediction, and nobody made it deliberately.

In [ ]:
enc = OneHotEncoder(handle_unknown="ignore").fit(
    housing[["ocean_proximity"]].query("ocean_proximity != 'ISLAND'"))
print("an unseen category encodes to:", enc.transform([["ISLAND"]]).toarray()[0])
print("sum:", enc.transform([["ISLAND"]]).toarray().sum(), "— and no warning")

## 7 · The test set. Once.

Everything so far used only training data. This is the first and last time the
test set is touched.

> **Prompt · the test set, once**
>
> **input** · the 4,128 held-out districts
>
> **output** · the test RMSE with a 95% bootstrap interval, beside the cross-validated estimate
>
> **constraint** · `method='percentile'` is NOT optional — scipy defaults to BCa, and we are claiming the percentile bootstrap. Name the estimator you mean

**Watch this prompt.**

* **Left open:** how to read the two numbers together. They agree within the interval, and a gap smaller than the interval is not evidence of anything.
* **The usual student version:** tuning after seeing this number. If you adjust hyperparameters to improve it you are fitting the test set, and the improvement will not generalise.
* **How you would catch it:** bootstrap the SQUARED ERRORS and take the square root of the interval, rather than bootstrapping the RMSE directly. The mean is what the bootstrap is good at; the square root comes after.

In [ ]:
from scipy import stats

final_pred = best.predict(X_test)
final_rmse = root_mean_squared_error(y_test, final_pred)

squared = (final_pred - y_test.values) ** 2
# method= is not optional: scipy defaults to BCa, and we are claiming the
# PERCENTILE bootstrap. Name the estimator you mean.
lo, hi = np.sqrt(stats.bootstrap([squared], np.mean,
                                 confidence_level=0.95, method="percentile",
                                 random_state=RANDOM_STATE).confidence_interval)

print(f"test RMSE  ${final_rmse:,.0f}")
print(f"95% interval  ${lo:,.0f} – ${hi:,.0f}")
print(f"\ncross-validated estimate was ${-search.best_score_:,.0f}")
print("The two agree within the interval. A gap smaller than the interval is "
      "not evidence of anything.")

**Do not tune now.** If you adjust hyperparameters to improve that number you
are fitting the test set, and the improvement will not generalise. The number
you have is the number you report.

## 8 · Red-team

Swap notebooks with the team beside you. Ten minutes. Five questions:

1. What touched the test set?
2. What was fitted, and on what? (`fit` and `transform` are different verbs)
3. What is the shape here?
4. What was dropped — rows, columns, NaNs? Count them.
5. What is the default I did not ask for?

Report what you **found**, not what you would have done differently.